# Runtime scaling of SciPy L-BFGS-B

This notebook investigates a possible runtime-scaling issue in `scipy.optimize.minimize(method="L-BFGS-B")`.

The question is deliberately about wall-clock overhead, not which method reaches the smallest objective value.

## Reproduction protocol

All solvers receive an analytic gradient, use L-BFGS memory `m=10`, run without callbacks, and have convergence tolerances disabled as far as their public interfaces allow. 

In [ ]:
import os
import platform
import sys
import time
from dataclasses import asdict, dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import minimize

from qnlab.parameter import LineParameter, NTRQNParameter
from qnlab.problem.base import BaseProblem
from qnlab.solver.qn_line import qn_line
from qnlab.solver.qn_ntrqn import qn_ntrqn
from qnlab.util.method import Method
from qnlab.problem.ill_quadratic import IllQuadraticProblem

ENVIRONMENT = {
    "python": sys.version.replace("\n", " "),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "platform": platform.platform(),
    "processor": platform.processor() or "not reported",
    "cpu_count": os.cpu_count(),
}
pd.Series(ENVIRONMENT, name="value").to_frame()

## Problem definition

We used the following two test problems to investigate the issue.

### Problem 1: A smooth zero-chain problem

The first one is the famous "zero-chain" problem. For dimension $n$, define

$$
f(x)=\frac12 x_1^2-x_1+\frac12\sum_{i=1}^{n-1}(x_i-x_{i+1})^2.
$$

This is a strictly convex, ill-conditioned quadratic with minimizer $x^*=(1,\ldots,1)$.
Taking `dimension > max_iterations` prevents the algorithm from simply exhausting the chain before the 10,000-iteration measurement.
Both the function and gradient below cost $\mathcal{O}(n)$ and allocate only a few vectors.


In [ ]:
class ZeroChainQuadratic(BaseProblem):
    def __init__(self, n: int):
        super().__init__(name="ZeroChainQuadratic", n=n, x0=np.zeros(n))
        self.x_opt = np.ones(n)

    def _f(self, x: np.ndarray) -> np.float64:
        differences = x[:-1] - x[1:]
        return np.float64(0.5 * x[0] ** 2 - x[0] + 0.5 * differences @ differences)

    def _g(self, x: np.ndarray) -> np.ndarray:
        differences = x[:-1] - x[1:]
        gradient = np.zeros_like(x)
        gradient[0] = x[0] - 1.0
        gradient[:-1] += differences
        gradient[1:] -= differences
        return gradient


# Sanity checks for the formula and zero-chain structure.
_problem = ZeroChainQuadratic(8)
assert np.isclose(_problem.f(_problem.x_opt, count=False), -0.5)
assert np.allclose(_problem.g(_problem.x_opt, count=False), 0.0)
assert np.array_equal(np.flatnonzero(_problem.g(_problem.x0, count=False)), [0])

## Problem 2: A diagonal quadratic problem

The second problem is a diagonal quadratic, which is close to `script/computation_time.py`.

The diagonal quadratic is

$$
f(x)=\frac12 x^\top \operatorname{diag}(1,\ldots,n)x.
$$

The initial point is $x_{\mathrm{init}}=(1,\ldots,1)$, and the minimizer is $x^*=(0,\ldots,0)$.


In [ ]:
# Sanity checks for the formula.
_problem = IllQuadraticProblem(8)
assert np.isclose(_problem.f(_problem.x_opt, count=False), 0.0)
assert np.allclose(_problem.g(_problem.x_opt, count=False), 0.0)
assert np.array_equal(_problem.g(_problem.x0, count=False), list(range(1, 8 + 1)))

## Benchmark implementation

The iteration cap controls termination by setting `ftol=gtol=0`.

In [ ]:
from collections.abc import Callable

MEMORY = 10
ITERATION_DIMENSION = 1200  # Must exceed max(ITERATION_BUDGETS).
ITERATION_BUDGETS = np.array([100, 300, 500, 700, 900, 1100])
REPEATS = 3

DIMENSION_ITERATIONS = 300
DIMENSIONS = np.array([500, 1000, 2000, 5000])
DIMENSION_REPEATS = 3

assert ITERATION_DIMENSION > ITERATION_BUDGETS.max()
assert DIMENSIONS.min() > DIMENSION_ITERATIONS
SOLVERS = ("SciPy L-BFGS-B", "qnlab qn_line", "qnlab qn_ntrqn")

In [ ]:
@dataclass
class TimingResult:
    solver: str
    dimension: int
    requested_iterations: int
    repeat: int
    elapsed_seconds: float
    iterations: int | None
    function_calls: int
    gradient_calls: int
    final_objective: float
    exit_status: str


def run_once(
    solver: str,
    dimension: int,
    max_iterations: int,
    repeat: int,
    memory: int = MEMORY,
    problem_factory: Callable[[int], BaseProblem] = ZeroChainQuadratic,
) -> TimingResult:
    problem = problem_factory(dimension)
    evaluation_limit = 50 * max_iterations + 100
    qnlab_options: dict[str, np.float64 | int] = {
        "m": memory,
        "max_iterations": max_iterations,
        "max_evaluations": evaluation_limit,
        "past": 0,
        "ftol": np.float64(0),
        "gtol": np.float64(0),
    }

    start = time.perf_counter()
    if solver == "SciPy L-BFGS-B":
        result = minimize(
            problem.f,
            problem.x0.copy(),
            jac=problem.g,
            method="L-BFGS-B",
            bounds=None,
            callback=None,
            options={
                "maxcor": memory,
                "maxiter": max_iterations,
                "maxfun": evaluation_limit,
                "ftol": 0.0,
                "gtol": 0.0,
                "maxls": 40,
            },
        )
        elapsed = time.perf_counter() - start
        iterations = int(result.nit)
        final_objective = float(result.fun)
        exit_status = str(result.message)
    elif solver == "qnlab qn_line":
        method = Method(base="Line", store="raw", secant="raw", update="bfgs")
        parameter = LineParameter(dimension, qnlab_options)
        code, final_objective, _ = qn_line(problem, parameter, method, callback=None)
        elapsed = time.perf_counter() - start
        iterations = max_iterations if "MAXIMUMITERATION" in str(code) else None
        exit_status = str(code)
    elif solver == "qnlab qn_ntrqn":
        method = Method(base="NTRQN", store="cautious", secant="damped", update="bfgs")
        parameter = NTRQNParameter(dimension, qnlab_options)
        code, final_objective, _ = qn_ntrqn(
            problem, parameter, method, callback=None, verbose=False
        )
        elapsed = time.perf_counter() - start
        iterations = max_iterations if "MAXIMUMITERATION" in str(code) else None
        exit_status = str(code)
    else:
        raise ValueError(f"Unknown solver: {solver}")

    return TimingResult(
        solver=solver,
        dimension=dimension,
        requested_iterations=max_iterations,
        repeat=repeat,
        elapsed_seconds=elapsed,
        iterations=iterations,
        function_calls=problem.call_f,
        gradient_calls=problem.call_g,
        final_objective=float(final_objective),
        exit_status=exit_status,
    )


def benchmark(
    cases: list[tuple[int, int]],
    repeats: int,
    problem_factory: Callable[[int], BaseProblem] = ZeroChainQuadratic,
    memory: int = MEMORY,
) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    # Untimed warm-up initializes imports and native code paths.
    for solver in SOLVERS:
        run_once(
            solver,
            dimension=600,
            max_iterations=5,
            repeat=-1,
            memory=memory,
            problem_factory=problem_factory,
        )

    for repeat in range(repeats):
        ordered_solvers = (
            SOLVERS[repeat % len(SOLVERS) :] + SOLVERS[: repeat % len(SOLVERS)]
        )
        for dimension, iterations in cases:
            for solver in ordered_solvers:
                measurement = run_once(
                    solver,
                    dimension,
                    iterations,
                    repeat,
                    memory=memory,
                    problem_factory=problem_factory,
                )
                rows.append(asdict(measurement))
                print(
                    f"repeat={repeat + 1}/{repeats}, n={dimension}, k={iterations}, "
                    f"m={memory}, "
                    f"{solver}: {measurement.elapsed_seconds:.3f} s"
                )
    return pd.DataFrame(rows)


def summarize(results: pd.DataFrame, *keys: str) -> pd.DataFrame:
    return results.groupby(["solver", *keys], as_index=False).agg(
        median_seconds=("elapsed_seconds", "median"),
        min_seconds=("elapsed_seconds", "min"),
        max_seconds=("elapsed_seconds", "max"),
        median_function_calls=("function_calls", "median"),
        median_gradient_calls=("gradient_calls", "median"),
        median_objective=("final_objective", "median"),
    )


def show_dimension_scaling(results: pd.DataFrame, title: str) -> pd.DataFrame:
    summary = summarize(results, "dimension")
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for solver, group in summary.groupby("solver"):
        ax.loglog(group["dimension"], group["median_seconds"], "o-", label=solver)
    ax.set(xlabel="Dimension n", ylabel="Median elapsed time [s]", title=title)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    display(summary)
    plt.show()
    return summary

## Experiment 1: elapsed time versus iteration budget

In [ ]:
iteration_cases = [(ITERATION_DIMENSION, int(k)) for k in ITERATION_BUDGETS]
iteration_results = benchmark(iteration_cases, repeats=REPEATS)

In [ ]:
iteration_summary = summarize(iteration_results, "requested_iterations")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for solver, group in iteration_summary.groupby("solver"):
    axes[0].plot(
        group["requested_iterations"], group["median_seconds"], "o-", label=solver
    )
    axes[1].plot(
        group["requested_iterations"],
        1e6 * group["median_seconds"] / group["requested_iterations"],
        "o-",
        label=solver,
    )
axes[0].set(
    xlabel="Iteration cap",
    ylabel="Median elapsed time [s]",
    title=f"End-to-end time (n={ITERATION_DIMENSION:,})",
)
axes[1].set(
    xlabel="Iteration cap",
    ylabel="Median time / iteration [µs]",
    title="Normalized iteration cost",
)
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.tight_layout()
display(iteration_summary)

## Experiment 2: elapsed time versus dimension

In [ ]:
dimension_cases = [(int(n), DIMENSION_ITERATIONS) for n in DIMENSIONS]
dimension_results = benchmark(dimension_cases, repeats=DIMENSION_REPEATS)
dimension_summary = show_dimension_scaling(
    dimension_results,
    f"Zero-chain ({DIMENSION_ITERATIONS} iterations, m={MEMORY})",
)

## Experiment 3: diagonal quadratic at large dimension

In [ ]:
ILL_QUADRATIC_ITERATIONS = 100
ILL_QUADRATIC_DIMENSIONS = np.array([100, 300, 1000, 3000, 10000, 30000])
ILL_QUADRATIC_REPEATS = 3

ill_quadratic_cases = [
    (int(n), ILL_QUADRATIC_ITERATIONS) for n in ILL_QUADRATIC_DIMENSIONS
]
ill_quadratic_results = benchmark(
    ill_quadratic_cases,
    repeats=ILL_QUADRATIC_REPEATS,
    problem_factory=IllQuadraticProblem,
)

In [ ]:
ill_quadratic_summary = summarize(ill_quadratic_results, "dimension")
reference_times = ill_quadratic_summary[
    ill_quadratic_summary["solver"] == "qnlab qn_line"
].set_index("dimension")["median_seconds"]
ill_quadratic_summary["ratio_to_qn_line"] = ill_quadratic_summary.apply(
    lambda row: row["median_seconds"] / reference_times.loc[row["dimension"]], axis=1
)

ill_quadratic_slopes = {}
ill_quadratic_large_n_slopes = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for solver, group in ill_quadratic_summary.groupby("solver"):
    slope = np.polyfit(np.log(group["dimension"]), np.log(group["median_seconds"]), 1)[
        0
    ]
    large_group = group[group["dimension"] >= 30_000]
    large_slope = np.polyfit(
        np.log(large_group["dimension"]), np.log(large_group["median_seconds"]), 1
    )[0]
    ill_quadratic_slopes[solver] = slope
    ill_quadratic_large_n_slopes[solver] = large_slope
    axes[0].loglog(
        group["dimension"],
        group["median_seconds"],
        "o-",
        label=f"{solver} (p={slope:.2f})",
    )
    axes[1].semilogx(group["dimension"], group["ratio_to_qn_line"], "o-", label=solver)
axes[0].set(
    xlabel="Dimension n",
    ylabel="Median elapsed time [s]",
    title=f"Ill-conditioned quadratic ({ILL_QUADRATIC_ITERATIONS} iterations)",
)
axes[1].axhline(1.0, color="black", linewidth=1, alpha=0.5)
axes[1].set(
    xlabel="Dimension n",
    ylabel="Median time / qn_line median time",
    title="Relative runtime",
)
for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
fig.tight_layout()
display(ill_quadratic_summary)
display(
    pd.DataFrame(
        {
            "all_dimensions_slope": ill_quadratic_slopes,
            "n_at_least_30000_slope": ill_quadratic_large_n_slopes,
        }
    )
)

## Experiment 4: L-BFGS memory $m=3$ versus $m=10$

In [ ]:
MEMORY_VALUES = (3, 10)
MEMORY_COMPARISON_REPEATS = 5  # Use 3 for a quick run.
MEMORY_COMPARISON_DIMENSIONS = np.array([100, 300, 1000, 3000, 10000, 30000])
MEMORY_COMPARISON_ITERATIONS = 100

memory_cases = [
    (int(n), MEMORY_COMPARISON_ITERATIONS) for n in MEMORY_COMPARISON_DIMENSIONS
]
memory_results = pd.concat(
    [
        benchmark(
            memory_cases,
            repeats=MEMORY_COMPARISON_REPEATS,
            problem_factory=IllQuadraticProblem,
            memory=memory,
        ).assign(memory=memory)
        for memory in MEMORY_VALUES
    ],
    ignore_index=True,
)

In [ ]:
memory_summary = summarize(memory_results, "memory", "dimension")

memory_ratio = memory_summary.pivot(
    index=["solver", "dimension"], columns="memory", values="median_seconds"
).reset_index()
memory_ratio["m3_over_m10"] = memory_ratio[3] / memory_ratio[10]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for (solver, memory), group in memory_summary.groupby(["solver", "memory"]):
    axes[0].loglog(
        group["dimension"],
        group["median_seconds"],
        "o-",
        label=f"{solver}, m={memory}",
    )
for solver, group in memory_ratio.groupby("solver"):
    axes[1].semilogx(group["dimension"], group["m3_over_m10"], "o-", label=solver)
axes[0].set(
    xlabel="Dimension n",
    ylabel="Median elapsed time [s]",
    title=f"Memory comparison ({MEMORY_COMPARISON_ITERATIONS} iterations)",
)
axes[1].axhline(1.0, color="black", linewidth=1, alpha=0.5)
axes[1].set(
    xlabel="Dimension n",
    ylabel="Median time at m=3 / median time at m=10",
    title="Effect of reducing L-BFGS memory",
)
for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize="small")
fig.tight_layout()
display(memory_summary)
display(memory_ratio)